# Step 8 — Mask-Based Anomaly Segmentation Baselines (EoMT)

Anomaly detection pipeline for **EoMT (DINOv2)** across the five datasets (**RA-21, RO-21, FS L&F, FS Static, and Road Anomaly**), evaluating the **MSP, MaxLogit, MaxEntropy, and RbA** methods on all three checkpoints (**COCO, Cityscapes, and fine-tuned**).

## Key Notes

- Convert model outputs into per-pixel anomaly maps using the native EoMT sliding-window pipeline:
  `window_imgs_semantic` → `to_per_pixel_logits_semantic` → `revert_window_logits_semantic`.
- A single forward pass per image is used to compute all anomaly scores.
- Input images must be provided in **uint8 format in the [0, 255] range** (the model internally performs `/255` scaling and normalization).
- `img_size` is automatically inferred from the checkpoint:
  - **Cityscapes** = 1024
  - **COCO** = 640
  - **Fine-tuned** = 640

## Imports

In [1]:
import subprocess, sys, os
import numpy as np
import glob as _glob, random
import gc
import torch
from torch.nn import functional as F
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import average_precision_score, roc_curve
from huggingface_hub import hf_hub_download, login
login()

## Environment setup (fix nump>=2) + repo

In [2]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpy>=2"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "lightning", "transformers", "timm", "torchmetrics", "scipy"], check=False)

# Ensure NumPy >= 2.x is available.
# If an older version is detected, restart the runtime so the updated environment is correctly loaded.
if int(np.__version__.split(".")[0]) < 2:
    print("NumPy version < 2 detected: automatically restarting the runtime...")
    print("   -> After the restart, begin again from THIS cell and run the notebook sequentially.")
    os.kill(os.getpid(), 9)
else:
    print("Environment ready | NumPy", np.__version__)

Environment ready | NumPy 2.0.1


## Import (autosufficient on the path)

In [5]:
EOMT_DIR = "../eomt" # Change the path to the eomt directory
assert os.path.isdir(EOMT_DIR), f"Repo not found in {EOMT_DIR}"
os.chdir(EOMT_DIR)
if EOMT_DIR not in sys.path:
    sys.path.insert(0, EOMT_DIR)

# fpr_at_95_tpr implemented as ood_metrics
def fpr_at_95_tpr(preds, labels):
    fpr, tpr, _ = roc_curve(labels, preds)
    if np.all(tpr < 0.95):
        return 0.0
    elif np.all(tpr >= 0.95):
        return float(min(fpr[i] for i, x in enumerate(tpr) if x >= 0.95))
    return float(np.interp(0.95, tpr, fpr))
print("Setup OK")

from models.vit import ViT
from models.eomt import EoMT
from training.mask_classification_semantic import MaskClassificationSemantic


seed = 42
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
torch.backends.cudnn.benchmark = True

device = 0


Setup OK


## Model Loader (Automatic `img_size` Inference from Checkpoint)

`build_model` downloads the checkpoint weights, inspects the shape of `pos_embed` (with an effective patch size of 16 in this ViT), and automatically infers the native input resolution: **1024** for Cityscapes checkpoints and **640** for COCO and fine-tuned checkpoints. This ensures that the positional embeddings match correctly and prevents input size mismatches when loading the model.

In [6]:
PRESET = "cityscapes"      # default for all cells
# None = checkpoint's NATIVE resolution (Cityscapes 1024, COCO/finetuned 640)
FORCE_IMG_SIZE = None
hugging_face_repo="gruppofundamentals1" # Change to the Hugging Face Hub repo name

PRESETS = {
    "cityscapes": dict(num_classes=19, num_q=100, num_blocks=3, img_size=(1024, 1024),
        repo_id=f"{hugging_face_repo}/cityscapes_semantic_eomt_base_640",
        ckpt_file="pytorch_model.bin", local_path=None),
    "coco": dict(num_classes=133, num_q=200, num_blocks=3, img_size=(640, 640),
        repo_id=f"{hugging_face_repo}/coco_panoptic_eomt_base_640",
        ckpt_file="pytorch_model.bin", local_path=None),
    "finetuned": dict(num_classes=19, num_q=100, num_blocks=3, img_size=(640, 640),
        repo_id=None, ckpt_file=None,
        local_path="./checkpoints/eomt_finetuned_cityscapes_step5_best_exp3.pt"), # Change to the path to the finetuned chechkpoints
}

def _resolve_state_dict(p, device):
    if p["local_path"]:
        raw = torch.load(p["local_path"], map_location=f"cuda:{device}", weights_only=False)
        sd = raw["state_dict"] if isinstance(raw, dict) and "state_dict" in raw else raw
    else:
        path = hf_hub_download(repo_id=p["repo_id"], filename=p["ckpt_file"])
        sd = torch.load(path, map_location=f"cuda:{device}", weights_only=True)
    return {k.replace("._orig_mod", ""): v for k, v in sd.items()
            if "criterion.empty_weight" not in k}

def _interp_pos_embed(sd, target_grid):
    """Interpoles the checkpoint's pos_embed to achieve a different resolution from the native one."""
    pe = sd.get("network.encoder.backbone.pos_embed")
    if pe is None or pe.dim() != 3:
        return sd
    g = round(pe.shape[1] ** 0.5)
    if g * g != pe.shape[1] or (g, g) == tuple(target_grid):
        return sd
    C = pe.shape[2]
    pe2 = pe.reshape(1, g, g, C).permute(0, 3, 1, 2)
    pe2 = F.interpolate(pe2, size=tuple(target_grid), mode="bicubic", align_corners=False)
    sd["network.encoder.backbone.pos_embed"] = pe2.permute(0, 2, 3, 1).reshape(1, target_grid[0] * target_grid[1], C)
    print(f"  pos_embed interpolated: {g}x{g} -> {target_grid[0]}x{target_grid[1]}")
    return sd

def build_model(preset_name, device=0):
    p = PRESETS[preset_name]
    sd = _resolve_state_dict(p, device)

    # native resolution from checkpoint (pos_embed = grid*grid, stride 16)
    img_size = tuple(p["img_size"])
    pe = sd.get("network.encoder.backbone.pos_embed")
    if pe is not None and pe.dim() == 3:
        grid = round(pe.shape[1] ** 0.5)
        if grid * grid == pe.shape[1]:
            img_size = (grid * 16, grid * 16)

    if FORCE_IMG_SIZE is not None:
        img_size = tuple(FORCE_IMG_SIZE)
        sd = _interp_pos_embed(sd, (img_size[0] // 16, img_size[1] // 16))

    print(f"[{preset_name}] img_size used: {img_size}")
    encoder = ViT(img_size=img_size, backbone_name="vit_base_patch14_reg4_dinov2")
    network = EoMT(encoder=encoder, num_classes=p["num_classes"],
                   num_q=p["num_q"], num_blocks=p["num_blocks"], masked_attn_enabled=False)
    model = MaskClassificationSemantic(network=network, img_size=img_size,
                num_classes=p["num_classes"], attn_mask_annealing_enabled=False).eval().to(device)

    missing, unexpected = model.load_state_dict(sd, strict=False)
    missing = [k for k in missing if "metrics." not in k and "criterion." not in k]
    print(f"  uploaded weights | num_classes={p['num_classes']} num_q={p['num_q']} | "
          f"missing(non-metric)={len(missing)} unexpected={len(unexpected)}")
    return model

## EoMT Output → Per-Pixel Map (Sliding-Window Inference)

For **MSP**, **MaxEntropy**, and **RbA**, per-pixel class probabilities are obtained as:

```text
probs[c, h, w] = Σ_q sigmoid(mask_q) · softmax(class_q)[c]
```

which corresponds to the `sem_seg` output of Mask2Former.

For **MaxLogit**, per-pixel logits are computed as:

```text
raw[c, h, w] = Σ_q sigmoid(mask_q) · class_logit_q[c]
```

The `temperature` parameter is used for temperature scaling before computing the anomaly scores.

In [29]:
@torch.no_grad()
def eomt_pixel_logits(model, img, device=0, temperature=1.0, return_raw=False):
    """img: tensor UINT8 [3,H,W] in [0,255]. Returns probs [C,H,W] (+ raw if required)."""
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        imgs = [img.to(device)]
        img_sizes = [im.shape[-2:] for im in imgs]
        crops, origins = model.window_imgs_semantic(imgs)              # sliding window EoMT
        mask_logits_per_layer, class_logits_per_layer = model(crops)
        mask_logits  = F.interpolate(mask_logits_per_layer[-1], model.img_size, mode="bilinear")
        class_logits = class_logits_per_layer[-1]

        if temperature == 1.0:
            crop_probs = model.to_per_pixel_logits_semantic(mask_logits, class_logits)
        else:
            crop_probs = torch.einsum("bqhw,bqc->bchw", mask_logits.sigmoid(),
                                      (class_logits / temperature).softmax(dim=-1)[..., :-1])
        probs = model.revert_window_logits_semantic(crop_probs, origins, img_sizes)[0].float()
        if not return_raw:
            return probs

        crop_raw = torch.einsum("bqhw,bqc->bchw", mask_logits.sigmoid(), class_logits[..., :-1])
        raw = model.revert_window_logits_semantic(crop_raw, origins, img_sizes)[0].float()
        return probs, raw

print("eomt_pixel_logits defined.")

eomt_pixel_logits defined.


## Post-Hoc Anomaly Scoring Methods (MSP, MaxLogit, MaxEntropy, RbA)

The anomaly scores are computed from the per-pixel outputs obtained through the mask–class aggregation process described above.

- **MSP (Maximum Softmax Probability):** anomaly score derived from the maximum class probability at each pixel.
- **MaxLogit:** anomaly score derived from the maximum aggregated class logit at each pixel.
- **MaxEntropy:** anomaly score based on the entropy of the per-pixel class probability distribution.
- **RbA:** implemented according to the requested formulation:

```text
RbA = -Σ_c tanh(sem_seg[c])
```

where `sem_seg` corresponds to the per-pixel class probabilities (`probs`) obtained from the mask–class combination. Higher values indicate a higher likelihood of anomaly.

In [30]:
def score_MSP(probs):
    p = probs.cpu().numpy()
    return 1.0 - np.max(p, axis=0)

def score_MaxLogit(raw):
    r = raw.cpu().numpy()
    return-np.max(r, axis=0)

def score_MaxEntropy(probs):
    p = probs.cpu().numpy()
    p = p / (p.sum(axis=0, keepdims=True) + 1e-10)

    entropy = -np.sum(p * np.log(p + 1e-12), axis=0)
    num_classes=p.shape[0]

    normalized_entropy=entropy/np.log(num_classes+1e-10)
    return normalized_entropy

def score_RbA(probs):
    # RbA = -sum_c tanh(sem_seg[c])  (sem_seg = probs). Anomalo = nessuna classe nota lo reclama.
    p = probs.cpu().numpy()
    return -np.sum(np.tanh(p), axis=0)

# Wrapper in stile get_RbA (stessa logica del tuo snippet, ma via la pipeline EoMT):
#   il 'sem_seg' del tuo codice corrisponde a probs di eomt_pixel_logits.
@torch.no_grad()
def get_RbA(model, x, device=0):
    probs = eomt_pixel_logits(model, x[0] if isinstance(x, (list, tuple)) else x, device=device)
    return -probs.tanh().sum(dim=0)         # tensor [H,W]

def compute_scores(probs, raw):
    return {"MSP": score_MSP(probs), "MaxLogit": score_MaxLogit(raw),
            "MaxEntropy": score_MaxEntropy(probs), "RbA": score_RbA(probs)}

METHODS = ["MSP", "MaxLogit", "MaxEntropy", "RbA"]
print("Defined:", METHODS)

Defined: ['MSP', 'MaxLogit', 'MaxEntropy', 'RbA']


## Ground Truth and Image Preprocessing

Ground-truth masks are converted to a unified format:

- **0** = In-Distribution (ID)
- **1** = Out-of-Distribution (OOD)
- **255** = Ignore

For **FS L&F**, the ground-truth encoding is detected automatically, supporting both the original LostAndFound annotation format and the unified ID/OOD representation.

Input images are processed in **uint8 format** and evaluated at a resolution of **(512, 1024)**.

In [32]:
EVAL_SIZE = (512, 1024)   # (H, W)

def get_pathGT(path):
    pathGT = path.replace("images", "labels_masks")
    if "RoadObstacle21" in pathGT or "RoadObsticle21" in pathGT:
        pathGT = pathGT.replace(".webp", ".png")
    if "fs_static" in pathGT:
        pathGT = pathGT.replace(".jpg", ".png")
    if "RoadAnomaly" in pathGT:
        pathGT = pathGT.replace(".jpg", ".png")
    return pathGT

def preprocess_gt(pathGT, mask):
    ood = np.array(mask)
    if "RoadAnomaly21" in pathGT:
        return ood
    if "RoadAnomaly" in pathGT and "RoadAnomaly21" not in pathGT:
        return np.where(ood == 2, 1, ood)
    if "LostAndFound" in pathGT or "FS_LostFound" in pathGT:
        u = np.unique(ood)
        if np.any((u > 1) & (u < 201)):          # classic encoding LAF: 0=ign,1=ID,2-200=OOD
            ood = np.where(ood == 0, 255, ood)
            ood = np.where(ood == 1, 0, ood)
            ood = np.where((ood > 1) & (ood < 201), 1, ood)
            return ood
        return ood                                # unified encoding 0=ID,1=OOD,255=ign
    return ood                                    # fs_static, RoadObstacle21: gia' 0/1

def load_img_uint8(path, size_hw=EVAL_SIZE):
    im = Image.open(path).convert("RGB").resize((size_hw[1], size_hw[0]), Image.BILINEAR)
    return torch.from_numpy(np.array(im)).permute(2, 0, 1).contiguous()   # uint8 [3,H,W]

def load_gt(path, size_hw=EVAL_SIZE):
    pathGT = get_pathGT(path)
    if not os.path.exists(pathGT):
        return None
    m = Image.open(pathGT).resize((size_hw[1], size_hw[0]), Image.NEAREST)
    return preprocess_gt(pathGT, m).astype(np.uint8)

print("GT + preprocessing defined. EVAL_SIZE =", EVAL_SIZE)

GT + preprocessing defined. EVAL_SIZE = (512, 1024)


## Dataset: robust discovery + diagnostics

In [39]:
BASE = "../Anomaly_Validation_Datasets/Validation_Dataset" # Change the Anomaly_Validation_Datasets path

DATASETS = [
    {"name": "RoadAnomaly21",  "label": "SMIYC RA-21",  "dirs": ["RoadAnomaly21"],                  "exts": ["png","jpg","jpeg","webp"]},
    {"name": "RoadObstacle21", "label": "SMIYC RO-21",  "dirs": ["RoadObstacle21","RoadObsticle21"],"exts": ["webp","png","jpg","jpeg"]},
    {"name": "FS_LostFound",   "label": "FS L&F",       "dirs": ["FS_LostFound_full","FS_LostFound","LostAndFound"], "exts": ["png","jpg","jpeg"]},
    {"name": "fs_static",      "label": "FS Static",    "dirs": ["fs_static","FS_Static"],           "exts": ["jpg","png","jpeg"]},
    {"name": "RoadAnomaly",    "label": "Road Anomaly", "dirs": ["RoadAnomaly"],                     "exts": ["jpg","png","jpeg"]},
]

def resolve_files(ds):
    for d in ds["dirs"]:
        for e in ds["exts"]:
            f = sorted(_glob.glob(os.path.join(BASE, d, "images", f"*.{e}")))
            if f:
                return f
    return []

# DIAGNOSTICA: file trovati, GT esistenti, valori GT grezzi vs dopo preprocessing
print(f"{'dataset':<16}{'img':>5}{'gt_ok':>7}   raw_GT_unique            -> post_unique")
for ds in DATASETS:
    files = resolve_files(ds)
    if not files:
        print(f"{ds['name']:<16}{'0':>5}   (no file found)"); continue
    gt_ok = sum(os.path.exists(get_pathGT(f)) for f in files[:30])
    pgt = get_pathGT(files[0])
    if os.path.exists(pgt):
        raw_u = np.unique(np.array(Image.open(pgt)))[:8]
        post_u = np.unique(preprocess_gt(pgt, Image.open(pgt)))
        print(f"{ds['name']:<16}{len(files):>5}{gt_ok:>7}   {str(raw_u):<24} -> {post_u}")
    else:
        print(f"{ds['name']:<16}{len(files):>5}{gt_ok:>7}   missing GT: {pgt}")

dataset           img  gt_ok   raw_GT_unique            -> post_unique
RoadAnomaly21      10     10   [  0   1 255]            -> [  0   1 255]
RoadObstacle21     30     30   [  0   1 255]            -> [  0   1 255]
FS_LostFound      100     30   [  0   1 255]            -> [  0   1 255]
fs_static          30     30   [  0   1 255]            -> [  0   1 255]
RoadAnomaly        60     30   [0 2]                    -> [0 1]


## Evaluation Across All Three Checkpoints

For each checkpoint preset, the evaluation pipeline performs the following steps:

1. Build and load the model from the selected checkpoint.
2. Evaluate the model on all five datasets using a single forward pass per image.
3. Release GPU memory after completion.
4. Proceed to the next checkpoint.

In [34]:
PRESETS_TO_RUN = ["cityscapes", "coco", "finetuned"]
all_results_by_ckpt = {}
ds_order = [d["name"] for d in DATASETS]
labels   = {d["name"]: d["label"] for d in DATASETS}

def _f(v): return f"{v*100:.1f}" if v is not None else "N/A"

def print_table(preset, res):
    print(f"\n=== EoMT [{preset}] ===  (RAWS: MSP, MaxLogit, MaxEntropy, RbA)")
    h1 = f"{'Method':<12}" + "".join(f"  {labels[d]:>22}" for d in ds_order)
    h2 = f"{'':12}" + "".join(f"  {'AuPRC':>10} {'FPR95':>10}" for _ in ds_order)
    print(h1); print(h2); print("-" * len(h2))
    for m in METHODS:
        row = f"{m:<12}"
        for d in ds_order:
            r = res.get(d, {}).get(m)
            row += f"  {(_f(r['auprc']) if r else 'N/A'):>10} {(_f(r['fpr95']) if r else 'N/A'):>10}"
        print(row)
    print("-" * len(h2) + "   (values in %)")

def evaluate_current_model(model):
    res = {ds["name"]: {} for ds in DATASETS}
    for ds in DATASETS:
        files = resolve_files(ds)
        if not files:
            print(f"  [SKIP] {ds['name']}: no file found"); continue
        gts, scores = [], {m: [] for m in METHODS}
        for path in tqdm(files, desc=ds["name"], leave=False):
            if os.path.isdir(path): continue
            gt = load_gt(path)
            if gt is None or 1 not in np.unique(gt): continue
            img = load_img_uint8(path)
            probs, raw = eomt_pixel_logits(model, img, device=device, return_raw=True)
            sc = compute_scores(probs, raw)
            gts.append(gt)
            for m in METHODS: scores[m].append(sc[m].astype(np.float32))
            del probs, raw; torch.cuda.empty_cache()
        if not gts:
            print(f"  [SKIP] {ds['name']}: no valid GT"); continue
        G = np.array(gts); om, im = (G == 1), (G == 0)
        for m in METHODS:
            S = np.array(scores[m])
            out = np.concatenate((S[im], S[om]))
            lab = np.concatenate((np.zeros(im.sum()), np.ones(om.sum())))
            res[ds["name"]][m] = {"auprc": average_precision_score(lab, out),
                                  "fpr95": fpr_at_95_tpr(out, lab)}
        r = res[ds["name"]]
        print(f"  {ds['name']:<16} " +
              " | ".join(f"{m} A={r[m]['auprc']*100:4.1f} F={r[m]['fpr95']*100:5.1f}" for m in METHODS))
        del gts, scores, G; gc.collect(); torch.cuda.empty_cache()
    return res

try:
    del model; gc.collect(); torch.cuda.empty_cache()
except NameError:
    pass

for preset in PRESETS_TO_RUN:
    print("=" * 64); print("CHECKPOINT:", preset); print("=" * 64)
    try:
        m = build_model(preset, device=device)
    except Exception as e:
        print(f"[SKIP {preset}] failed upload: {type(e).__name__}: {e}")
        continue
    all_results_by_ckpt[preset] = evaluate_current_model(m)
    print_table(preset, all_results_by_ckpt[preset])
    del m; gc.collect(); torch.cuda.empty_cache()

print("\nDone")

CHECKPOINT: cityscapes
[cityscapes] img_size used: (1024, 1024)


c:\Users\user\anaconda3\envs\pytorch_gpu\Lib\site-packages\lightning\pytorch\utilities\parsing.py:213: Attribute 'network' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['network'])`.


  uploaded weights | num_classes=19 num_q=100 | missing(non-metric)=0 unexpected=0


  RoadAnomaly21    MSP A=68.6 F= 14.4 | MaxLogit A=62.8 F= 25.7 | MaxEntropy A=65.2 F= 31.5 | RbA A=65.1 F= 95.6


  RoadObstacle21   MSP A=84.1 F=  4.6 | MaxLogit A=64.7 F=100.0 | MaxEntropy A=73.8 F= 13.5 | RbA A=80.8 F= 99.9


...

  RoadAnomaly21    MSP A=28.7 F=100.0 | MaxLogit A=32.3 F= 93.2 | MaxEntropy A=75.0 F= 33.4 | RbA A=12.2 F= 99.9


  RoadObstacle21   MSP A=56.6 F= 52.8 | MaxLogit A=59.1 F= 95.6 | MaxEntropy A=78.1 F= 32.3 | RbA A=40.4 F=100.0


  FS_LostFound     MSP A=37.0 F= 14.8 | MaxLogit A=17.0 F= 79.7 | MaxEntropy A=30.6 F= 64.3 | RbA A=37.3 F= 87.7


  fs_static        MSP A=46.0 F= 95.8 | MaxLogit A=49.0 F= 93.9 | MaxEntropy A=45.2 F= 31.1 | RbA A=37.5 F= 98.0


  RoadAnomaly      MSP A=24.9 F= 75.3 | MaxLogit A=69.3 F= 70.0 | MaxEntropy A=23.5 F= 65.2 | RbA A=16.2 F= 93.8

=== EoMT [finetuned] ===  (RAWS: MSP, MaxLogit, MaxEntropy, RbA)
Method                   SMIYC RA-21             SMIYC RO-21                  FS L&F               FS Static            Road Anomaly
                   AuPRC      FPR95       AuPRC      FPR95       AuPRC      FPR95       AuPRC      FPR95       AuPRC      FPR95
-------------------------------------------------------------------------------------------------------------------------------
MSP                 28.7      100.0        56.6       52.8        37.0       14.8        46.0       95.8        24.9       75.3
MaxLogit            32.3       93.2        59.1       95.6        17.0       79.7        49.0       93.9        69.3       70.0
MaxEntropy          75.0       33.4        78.1       32.3        30.6       64.3        45.2       31.1        23.5       65.2
RbA                 12.2       99.9        40.4 

## Tabels (one for each checkpoint)

In [35]:
ds_order = [d["name"] for d in DATASETS]
labels = {d["name"]: d["label"] for d in DATASETS}
def _f(v): return f"{v*100:.1f}" if v is not None else "N/A"

for preset, res in all_results_by_ckpt.items():
    print(f"\n=== EoMT [{preset}] ===")
    sub = f"{'Method':<12}" + "".join(f"  {labels[d]+' AuPRC':>18} {'FPR95':>7}" for d in ds_order)
    print(sub); print("-" * len(sub))
    for m in METHODS:
        row = f"{m:<12}"
        for d in ds_order:
            r = res.get(d, {}).get(m)
            row += f"  {(_f(r['auprc']) if r else 'N/A'):>18} {(_f(r['fpr95']) if r else 'N/A'):>7}"
        print(row)
    print("-" * len(sub))


=== EoMT [cityscapes] ===
Method         SMIYC RA-21 AuPRC   FPR95   SMIYC RO-21 AuPRC   FPR95        FS L&F AuPRC   FPR95     FS Static AuPRC   FPR95  Road Anomaly AuPRC   FPR95
--------------------------------------------------------------------------------------------------------------------------------------------------------
MSP                         68.6    14.4                84.1     4.6                16.8     9.9                62.5    43.4                67.4    25.2
MaxLogit                    62.8    25.7                64.7   100.0                17.1    16.2                64.1    47.4                63.3    38.7
MaxEntropy                  65.2    31.5                73.8    13.5                11.5    25.7                19.6    52.5                48.6    43.9
RbA                         65.1    95.6                80.8    99.9                16.5     5.8                64.4    75.9                66.6    21.5
-------------------------------------------------------

## Temperature Scaling (MSP) — All Checkpoints and All Validation Datasets

Apply **temperature scaling** to the **MSP** anomaly score for the **Cityscapes**, **COCO**, and **fine-tuned** checkpoints across **all five validation datasets**, and determine the **best temperature `T`**, defined as the value that maximizes **AuPRC**.

### Implementation Strategy

Following the recommended optimization strategy, each image is processed with **a single forward pass** through the model. The resulting `mask_logits` and `class_logits` are cached and subsequently recombined for all candidate temperature values, avoiding repeated model inference for each `T`.

For a given temperature `T`, class logits are rescaled before the mask–class aggregation step, and the MSP score is computed from the resulting per-pixel probability distribution.

### Consistency with the Baseline

The special case **`T = 1.0`** reproduces the standard MSP baseline exactly, using the same EoMT processing pipeline (`to_per_pixel_logits_semantic`). This guarantees that temperature scaling can be evaluated directly against the original MSP results without introducing any additional changes to the inference procedure.

In [36]:
@torch.no_grad()
def eomt_msp_multiT(model, img, temps, device=0):
    """img: tensor UINT8 [3,H,W] in [0,255].
    Returs dict {T: score_MSP [H,W] np.float32}. Only one forward, many T."""
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        imgs = [img.to(device)]
        img_sizes = [im.shape[-2:] for im in imgs]
        crops, origins = model.window_imgs_semantic(imgs)                 # sliding window EoMT
        mask_logits_per_layer, class_logits_per_layer = model(crops)
        mask_logits  = F.interpolate(mask_logits_per_layer[-1], model.img_size, mode="bilinear")
        class_logits = class_logits_per_layer[-1]
        mask_sig = mask_logits.sigmoid()                                  # computed only one time

        out = {}
        for T in temps:
            if T == 1.0:
                # same as baseline MSP
                crop_probs = model.to_per_pixel_logits_semantic(mask_logits, class_logits)
            else:
                cls = (class_logits / T).softmax(dim=-1)[..., :-1]
                crop_probs = torch.einsum("bqhw,bqc->bchw", mask_sig, cls)
            probs = model.revert_window_logits_semantic(crop_probs, origins, img_sizes)[0].float()
            out[float(T)] = (1.0 - probs.max(dim=0).values).cpu().numpy().astype(np.float32)
    return out

print("eomt_msp_multiT defined (1 forward -> MSP for all T).")

eomt_msp_multiT defined (1 forward -> MSP for all T).


In [40]:
TEMPS_REQUIRED = [0.5, 0.75, 1.0, 1.1]
TEMPS_SEARCH   = [0.1, 0.25, 0.5, 0.75, 0.9, 1.0, 1.1, 1.25, 1.5, 1.75, 2.0]
TEMPS = sorted(set(TEMPS_REQUIRED) | set(TEMPS_SEARCH))

PRESETS_FOR_TEMP = ["cityscapes", "coco", "finetuned"]

# temp_results[preset][ds_name][T] = {"auprc":.., "fpr95":..}
temp_results = {}

try:
    del model; gc.collect(); torch.cuda.empty_cache()
except NameError:
    pass

for preset in PRESETS_FOR_TEMP:
    print("=" * 64); print("TEMP SCALING (MSP) — CHECKPOINT:", preset); print("=" * 64)
    try:
        m = build_model(preset, device=device)
    except Exception as e:
        print(f"[SKIP {preset}] upload failed: {type(e).__name__}: {e}")
        continue
    temp_results[preset] = {}

    for ds in DATASETS:
        files = resolve_files(ds)
        if not files:
            print(f"  [SKIP] {ds['name']}: no file found"); continue

        # accumulo per T gli score dei pixel inlier (gt==0) e outlier (gt==1)
        id_sc  = {T: [] for T in TEMPS}
        ood_sc = {T: [] for T in TEMPS}

        for path in tqdm(files, desc=f"{preset}/{ds['name']}", leave=False):
            if os.path.isdir(path): continue
            gt = load_gt(path)
            if gt is None or 1 not in np.unique(gt): continue
            img = load_img_uint8(path)
            msp_byT = eomt_msp_multiT(m, img, TEMPS, device=device)        # Only ONE forward, all T
            im, om = (gt == 0), (gt == 1)
            for T in TEMPS:
                s = msp_byT[float(T)]
                id_sc[T].append(s[im]); ood_sc[T].append(s[om])
            del msp_byT, img; torch.cuda.empty_cache()

        if not ood_sc[TEMPS[0]]:
            print(f"  [SKIP] {ds['name']}: no valid GT"); continue

        temp_results[preset][ds["name"]] = {}
        for T in TEMPS:
            inl  = np.concatenate(id_sc[T])
            outl = np.concatenate(ood_sc[T])
            preds = np.concatenate([inl, outl])
            lab   = np.concatenate([np.zeros(len(inl)), np.ones(len(outl))])
            temp_results[preset][ds["name"]][T] = {
                "auprc": average_precision_score(lab, preds),
                "fpr95": fpr_at_95_tpr(preds, lab),
            }
        del id_sc, ood_sc; gc.collect(); torch.cuda.empty_cache()

        byT   = temp_results[preset][ds["name"]]
        bestT = max(byT, key=lambda t: byT[t]["auprc"])
        print(f"  {ds['name']:<16} best t = {bestT:<5}  "
              f"AuPRC {byT[bestT]['auprc']*100:5.2f}%  (T=1.0: {byT[1.0]['auprc']*100:5.2f}%) | "
              f"FPR95 {byT[bestT]['fpr95']*100:5.2f}%")

    del m; gc.collect(); torch.cuda.empty_cache()

TEMP SCALING (MSP) — CHECKPOINT: cityscapes
[cityscapes] img_size used: (1024, 1024)


c:\Users\user\anaconda3\envs\pytorch_gpu\Lib\site-packages\lightning\pytorch\utilities\parsing.py:213: Attribute 'network' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['network'])`.


  uploaded weights | num_classes=19 num_q=100 | missing(non-metric)=0 unexpected=0


  RoadAnomaly21    best t = 2.0    AuPRC 70.42%  (T=1.0: 68.62%) | FPR95 12.56%


  RoadObstacle21   best t = 0.75   AuPRC 84.36%  (T=1.0: 84.13%) | FPR95  1.10%


...

  RoadAnomaly21    best t = 0.1    AuPRC 42.94%  (T=1.0: 28.71%) | FPR95 97.28%


  RoadObstacle21   best t = 0.1    AuPRC 63.48%  (T=1.0: 56.58%) | FPR95 36.07%


  FS_LostFound     best t = 0.5    AuPRC 38.18%  (T=1.0: 37.02%) | FPR95 13.86%


  fs_static        best t = 0.9    AuPRC 45.98%  (T=1.0: 45.97%) | FPR95 96.00%


  RoadAnomaly      best t = 1.75   AuPRC 29.89%  (T=1.0: 24.92%) | FPR95 83.22%


In [41]:
# Tabel with raws MSP, MSP(t=0.5/0.75/1.1), MSP(best t) for all 5 dataset.
ds_order = [d["name"] for d in DATASETS]
labels   = {d["name"]: d["label"] for d in DATASETS}
def _ff(v): return f"{v*100:.1f}" if v is not None else "N/A"

REPORT_ROWS = [("MSP",          1.0),
               ("MSP(t=0.5)",   0.5),
               ("MSP(t=0.75)",  0.75),
               ("MSP(t=1.1)",   1.1)]

best_t_overall = {}   # best average t

for preset, res in temp_results.items():
    print(f"\n=== EoMT [{preset}] — Temperature scaling on MSP ===")
    head = f"{'Method':<14}" + "".join(f"  {labels[d]+' AuPRC':>18} {'FPR95':>7}" for d in ds_order)
    print(head); print("-" * len(head))

    for name, T in REPORT_ROWS:
        row = f"{name:<14}"
        for d in ds_order:
            r = res.get(d, {}).get(T)
            row += f"  {(_ff(r['auprc']) if r else 'N/A'):>18} {(_ff(r['fpr95']) if r else 'N/A'):>7}"
        print(row)

    # for each dataset, the T che mmaximizes the AuPRC
    row = f"{'MSP (best t)':<14}"; best_per_ds = []
    for d in ds_order:
        byT = res.get(d, {})
        if not byT:
            row += f"  {'N/A':>18} {'N/A':>7}"; best_per_ds.append((d, None)); continue
        bt = max(byT, key=lambda t: byT[t]["auprc"])
        r  = byT[bt]
        row += f"  {_ff(r['auprc']):>18} {_ff(r['fpr95']):>7}"; best_per_ds.append((d, bt))
    print(row); print("-" * len(head))
    print("  best t for dataset: " +
          ", ".join(f"{labels[d]}={bt}" for d, bt in best_per_ds if bt is not None))

    common_T = None
    for d in ds_order:
        if res.get(d):
            common_T = set(res[d].keys()) if common_T is None else (common_T & set(res[d].keys()))
    if common_T:
        def _mean_auprc(T):
            vals = [res[d][T]["auprc"] for d in ds_order if res.get(d)]
            return sum(vals) / len(vals)
        bt_glob = max(common_T, key=_mean_auprc)
        best_t_overall[preset] = bt_glob
        print(f"  best global t (AuPRC maximum average): T = {bt_glob}  "
              f"(mean AuPRC {_mean_auprc(bt_glob)*100:.2f}% vs {_mean_auprc(1.0)*100:.2f}% a T=1.0)")


=== EoMT [cityscapes] — Temperature scaling on MSP ===
Method           SMIYC RA-21 AuPRC   FPR95   SMIYC RO-21 AuPRC   FPR95        FS L&F AuPRC   FPR95     FS Static AuPRC   FPR95  Road Anomaly AuPRC   FPR95
----------------------------------------------------------------------------------------------------------------------------------------------------------
MSP                           68.6    14.4                84.1     4.6                16.8     9.9                62.5    43.4                67.4    25.2
MSP(t=0.5)                    66.1    29.4                83.9     1.4                15.1    11.9                58.2    41.1                65.1    22.7
MSP(t=0.75)                   68.1    16.8                84.4     1.1                16.4    11.9                60.5    41.4                66.9    24.8
MSP(t=1.1)                    68.9    13.7                83.6     6.3                16.9     9.7                63.0    45.6                67.4    25.1
MSP (best t)  